# Acoplamento Hidráulico-Mecânico
### Estudo da influência da membrana elástica na rede hidráulica
#### Disciplina: SME0602 — Motores Numéricos para Simulação em Engenharia
#### Professor: Roberto F. Ausas
#### Grupo 3: 
* #### Beatriz Cosimatti
* #### Cecilia Queiroz
* #### Gabriel Zago
* #### Matheus Buzzon
* #### Pedro Vale
* #### Victor Silva


In [ ]:
# Importações!!!

import sys
import os
import importlib

root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if root not in sys.path:
    sys.path.insert(0, root)

import mechanic_hydraulic
importlib.reload(mechanic_hydraulic)
import env
importlib.reload(env)
from plotting import plot_relaxamento_problema3
config = env.CONFIG_MH

##### Para atingir um modelo computacional realista, é necessário, primeiramente, analisar a matemática que rege o problema. Portanto, como primeiro passo, deve-se deduzir a equação que molda o problema:

$$M\left(\frac{v^{n+1} - v^n}{\delta t}\right) + \left(\beta M + \frac{h}{2}U^T A^{-1}U\right)v^{n+1} + Kw^{n+1} = U^T A^{-1}b^{n+1}$$

##### As equações mecânicas e hidráulicas são resolvidas simultaneamente por meio de um sistema global:

$$
\mathbf{A}_{\mathrm{global}} \, \mathbf{x}^{n+1} = \mathbf{b}
$$

##### Que pode ser escrito como:

$$
\begin{bmatrix}
\frac{1}{\delta t}I & -I & 0 \\
K & \frac{1}{\delta t}M + D & -U^T \\
0 & \frac{h}{2}U & A
\end{bmatrix}
\begin{bmatrix}
w^{n+1} \\ v^{n+1} \\ p^{n+1}
\end{bmatrix}
=
\begin{bmatrix}
\frac{1}{\delta t}w^n \\
\frac{1}{\delta t}Mv^n \\
b^{n+1}
\end{bmatrix}
$$
 
$$\frac{1}{\delta t}w^{n+1} - v^{n+1} = \frac{1}{\delta t}w^n \tag{I}$$
 
$$Kw^{n+1} + \left(\frac{1}{\delta t}M + D\right)v^{n+1} - U^T p^{n+1} = \frac{1}{\delta t}Mv^n \tag{II}$$
 
$$\frac{h}{2}Uv^{n+1} + Ap^{n+1} = b^{n+1} \tag{III}$$
 
##### Passo 1: Isolar $p^{n+1}$ a partir da equação (III)
 
$$Ap^{n+1} = b^{n+1} - \frac{h}{2}Uv^{n+1}$$
 
$$p^{n+1} = A^{-1}\left(b^{n+1} - \frac{h}{2}Uv^{n+1}\right)$$
 
##### Passo 2: Isolar $w^{n+1}$ a partir da equação (I)
 
$$\frac{1}{\delta t}w^{n+1} = \frac{1}{\delta t}w^n + v^{n+1}$$
 
$$w^{n+1} = w^n + \delta t\, v^{n+1}$$
 
##### Passo 3: Substituir $p^{n+1}$ na equação (II)
 
$$Kw^{n+1} + \left(\frac{1}{\delta t}M + D\right)v^{n+1} - U^T\left[A^{-1}\left(b^{n+1} - \frac{h}{2}Uv^{n+1}\right)\right] = \frac{1}{\delta t}Mv^n$$
 
$$Kw^{n+1} + \frac{1}{\delta t}Mv^{n+1} + Dv^{n+1} - U^T A^{-1}b^{n+1} + \frac{h}{2}U^T A^{-1}Uv^{n+1} = \frac{1}{\delta t}Mv^n$$
 
##### Passo 4: Agrupar termos em $v^{n+1}$ e usar $D = \beta M$
 
$$\frac{1}{\delta t}Mv^{n+1} - \frac{1}{\delta t}Mv^n + \left(D + \frac{h}{2}U^T A^{-1}U\right)v^{n+1} + Kw^{n+1} = U^T A^{-1}b^{n+1}$$
 
$$M\left(\frac{v^{n+1} - v^n}{\delta t}\right) + \left(\beta M + \frac{h}{2}U^T A^{-1}U\right)v^{n+1} + Kw^{n+1} = U^T A^{-1}b^{n+1}$$

##### Essa formulação caracteriza um acoplamento monolítico, pois todas as variáveis do sistema são calculadas simultaneamente em cada passo temporal. 
 


##### Como existe um forte acoplamento entre o comportamento mecânico da membrana e o escoamento hidráulico, optou-se pela utilização de um método de integração temporal implícito, garantindo maior estabilidade numérica durante a simulação.

#### O Método de Euler Implícito
##### O método de Euler Implícito aproxima a derivada temporal utilizando uma diferença regressiva, avaliando a função no instante futuro t<sup>n+1. Assim: 
$$
\frac{d\mathbf{x}}{dt}
\approx
\frac{\mathbf{x}^{\,n+1}-\mathbf{x}^{\,n}}
{\Delta t}.
\tag{11}
$$

##### Substituindo essa aproximação na equacão diferencial obtém-se
$$
\mathbf{x}^{\,n+1}
=
\mathbf{x}^{\,n}
+
\Delta t\,
f(\mathbf{x}^{\,n+1},t^{\,n+1}),
\tag{13}
$$

##### Observa-se que as incógnitas do instante futuro aparecem em ambos os lados da equação, caracterizando um método implícito. Dessa forma, torna-se necessário resolver um sistema linear a cada passo de tempo para determinar o estado da solução.

![img problema 1](problema1.jpeg)

##### O problema de evolução temporal foi resolvido considerando os passos de tempo adimensionais: 𝛿𝑡 = 0.00625, 0.0125, 0.025 e 0.05 dentro do intervalo de tempo [0, 12]. Para a membrana elástica, utilizamos ambas as discretizações espaciais de (51, 51) e (101, 101) e para a rede hidráulica, analisamos para diferentes pressões na entrada: 𝑝inlet = 5 × 10^3 Pa, 10^4 Pa e 2 × 10^4 Pa.

In [ ]:
simulador = mechanic_hydraulic.MechanicHydraulic(config)
todos_resultados = simulador.resolver_todos_cenarios(print_info=False)

##### Deslocamento Centro

![Deslocamento Centro P 5e+03](P2/deslocamento_centro_P_5.0e+03.png)

![Deslocamento Centro P 1e+04](P2/deslocamento_centro_P_1.0e+04.png)

![Deslocamento Centro P 2e+04](P2/deslocamento_centro_P_2.0e+04.png)


##### Perfil Membrana

![Perfil Membrana P 5e+03](P2/perfil_membrana_P_5.0e+03.png)

![Perfil Membrana P 1e+04](P2/perfil_membrana_P_1.0e+04.png)

![Perfil Membrana P 2e+04](P2/perfil_membrana_P_2.0e+04.png)

##### Potência

![Potência P 5e+03](P2/potencia_P_5.0e+03.png)

![Potência P 1e+04](P2/potencia_P_1.0e+04.png)

![Potência P 2e+04](P2/potencia_P_2.0e+04.png)

##### Pressão Outlet

![Pressão Outlet P 5e+03](P2/pressao_outlet_P_5.0e+03.png)

![Pressão Outlet P 1e+04](P2/pressao_outlet_P_1.0e+04.png)

![Pressão Outlet P 2e+04](P2/pressao_outlet_P_2.0e+04.png)

##### Vazão Outlet

![Vazão Outlet P 5e+03](P2/vazao_outlet_P_5.0e+03.png)
![Vazão Outlet P 1e+04](P2/vazao_outlet_P_1.0e+04.png)
![Vazão Outlet P 2e+04](P2/vazao_outlet_P_2.0e+04.png)

##### Volume Reservatório

![Volume Reservatório P 5e+03](P2/volume_reservatorio_P_5.0e+03.png)
![Volume Reservatório P 1e+04](P2/volume_reservatorio_P_1.0e+04.png)
![Volume Reservatório P 2e+04](P2/volume_reservatorio_P_2.0e+04.png)

##### A partir das configurações obtidas ao final da simulação do item anterior, estudamos o cenário em que a pressão de entrada 𝑝 inlet decaiu instantaneamente para zero. O sistema continuou evoluindo a partir desse estado transiente até atingir um novo estado de equilíbrio estático

In [ ]:
#Roda o caso base uma vez com as restrições do enunciado para gerar o ponto de partida (estado inflado)
estado_inflado_ex2 = simulador.resolver_caso_base(
    N=(51, 51),
    dt=0.025,
    tempo_final=config["TIME_END"], 
    pressao_inlet=config["INLET_PRESSURE"], 
    largura_canal=config["CHANNEL_WIDTH"],
    print_info=False
)

resultado_problema3 = simulador.resolver_relaxamento(
    estado_inicial_ex2=estado_inflado_ex2,
    dt=0.025,
    tempo_final=12.0, 
    largura_canal=config["CHANNEL_WIDTH"],
    print_info=False
)
plot_relaxamento_problema3(resultado_problema3)

##### Depois, medimos a frequência de oscilação transiente do sistema considerando...

##### Pressão nula na entrada:

In [ ]:
config["N"] = (101, 101) 

solver_p4 = mechanic_hydraulic.MH_Problema4(config)
solver_p4.resolver_P4(dt=0.0125, tempo_final=12.0)

##### Pressão na entrada: $$p_{\text{inlet}}(t) = 5000\cos(\omega_3 t) \ \text{Pa}$$

In [ ]:
solver_p5 = mechanic_hydraulic.MH_Problema5(config)
solver_p5.resolver_P5();

##### A abordagem adotada se mostrou adequada para representar a interação física dos sistemas hidráulico e mecânico, proporcionando uma estabilidade numérica, através do método de Euler Implícito, e eficiência computacional durante as simulações, através das matrizes esparsas e da fatoração LU. Portanto, os métodos numéricos utilizados atenderam os objetivos propostos, o que permitiu a obtenção de bons resultados das variáveis de interesse e do sistema como um todo.